# 4. Análisis univariado

**Qué hacemos:** describir cada variable por separado, según su tipo. Numéricas: media, mediana, desviación, percentiles, IQR,
asimetría, curtosis, valores atípicos, histogramas y diagramas de caja. Categóricas: cardinalidad, frecuencias, categorías raras
y faltantes.

**Por qué:** las decisiones de preprocesamiento (transformaciones, agrupación de categorías, codificación) salen de aquí.
Todo usa solo el conjunto de entrenamiento.

In [ ]:
from pathlib import Path
import re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while not (project_root / "_config.yml").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("No se encontró _config.yml. Ejecuta el notebook dentro de la carpeta del libro.")
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for _folder in ("tables", "figures"):          # se crean si no existen
    (project_root / _folder).mkdir(exist_ok=True)

from src.project_utils import *

configure_plots()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)
TABLES = project_root / "tables"

def md_out(text):
    display(Markdown(text))

train = load_train()
frame = prepare_frame(train)
num_cols, cat_cols = column_types(frame)
y = frame[TARGET]

## Variables numéricas

Además de las ocho variables de conteo, se agrega `age_mid`, el punto medio del rango de edad (`[50-60)` → 55). Los identificadores
codificados (tipo de ingreso, destino al alta y origen del ingreso) **se tratan como categorías**, no como cantidades.

In [ ]:
resumen_num = numeric_summary(frame[num_cols])
resumen_num.to_csv(TABLES / "estadisticas_numericas.csv")
display(resumen_num.round(2))
sesgadas = resumen_num["asimetría"].sort_values(ascending=False)
ceros = resumen_num["ceros_%"].sort_values(ascending=False)
asim = sesgadas[sesgadas > 1]
muchos_ceros = ceros[ceros > 30]
md_out(f"- Variables con asimetría > 1: {', '.join(f'**{v}** ({es(asim[v], 2)})' for v in asim.index) if len(asim) else 'ninguna'}.\n"
       f"- Variables con más de 30 % de ceros: {', '.join(f'**{v}** ({es(muchos_ceros[v], 1)} %)' for v in muchos_ceros.index) if len(muchos_ceros) else 'ninguna'}.\n\n"
       + ("Son variables de conteo con **cola larga hacia la derecha**: la mayoría de los pacientes tiene valores bajos y unos pocos valores muy altos. "
          "Esto sugiere una transformación logarítmica (se decide en el capítulo 8)." if len(asim) else
          "Ninguna requiere transformación por asimetría."))

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, c in zip(axes.ravel(), num_cols):
    s = frame[c].dropna()
    sns.histplot(s, discrete=s.nunique() <= 30, bins=40, ax=ax, color="#4c78a8")
    ax.set_title(c); ax.set_xlabel("")
for ax in axes.ravel()[len(num_cols):]:
    ax.axis("off")
plt.suptitle("Histogramas de las variables numéricas (entrenamiento)", y=1.01)
plt.tight_layout(); save_figure(fig, "04_histogramas_numericas.png"); plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 7))
for ax, c in zip(axes.ravel(), num_cols):
    sns.boxplot(x=frame[c], ax=ax, color="#9ecae9", fliersize=1.5)
    ax.set_title(c); ax.set_xlabel("")
for ax in axes.ravel()[len(num_cols):]:
    ax.axis("off")
plt.suptitle("Diagramas de caja (los puntos son valores atípicos por la regla del IQR)", y=1.01)
plt.tight_layout(); save_figure(fig, "04_boxplots_numericas.png"); plt.show()

### ¿Siguen una distribución normal? (pruebas de normalidad, con cautela)

El enunciado pide pruebas de normalidad cuando sean relevantes. Aquí **no se usan como criterio de decisión**: con más de 80.000 encuentros de entrenamiento, pruebas como Shapiro–Wilk, D'Agostino–Pearson o Kolmogorov–Smirnov rechazan la normalidad ante desviaciones mínimas y sin relevancia práctica (el p-valor refleja sobre todo el tamaño de la muestra). Además, son variables de **conteo discretas**, que por construcción no pueden ser normales. Por eso la forma de cada distribución se juzga con la **asimetría**, la **curtosis** (tabla anterior) y con gráficos: los histogramas, los diagramas de caja y los **gráficos Q-Q** frente a la normal que se muestran a continuación. Ningún método del libro (regresión logística, bootstrap, pruebas no paramétricas) exige normalidad de las predictoras.

In [ ]:
from scipy import stats as st
qq_vars = [c for c in ["time_in_hospital", "num_medications", "number_inpatient"] if c in frame.columns]
fig, axes = plt.subplots(1, len(qq_vars), figsize=(4.6 * len(qq_vars), 4.2))
for ax, c in zip(np.atleast_1d(axes), qq_vars):
    s = frame[c].dropna().to_numpy()
    (osm, osr), (pendiente, intercepto, _) = st.probplot(s, dist="norm")
    ax.plot(osm, osr, "o", ms=2, alpha=0.3, color="#4c78a8", rasterized=True)
    ax.plot(osm, pendiente * osm + intercepto, color="#e45756", lw=1.5, label="referencia normal")
    ax.set_title(f"{c}\nasimetría {es(pd.Series(s).skew(), 2)} · curtosis {es(pd.Series(s).kurt(), 2)}")
    ax.set_xlabel("cuantiles teóricos N(0, 1)"); ax.set_ylabel("cuantiles observados")
np.atleast_1d(axes)[0].legend(loc="upper left")
plt.suptitle("Gráficos Q-Q frente a la distribución normal (entrenamiento)", y=1.02)
plt.tight_layout(); save_figure(fig, "04_qqplots_numericas.png"); plt.show()
md_out("**Interpretación.** Ninguna de las tres sigue la recta: los escalones horizontales son propios de conteos discretos. "
       "`time_in_hospital` está acotada entre 1 y 14 días, por lo que ambos extremos se aplanan; `num_medications` se ajusta en el centro, "
       "pero su cola derecha es más pesada que la normal; `number_inpatient` se concentra en 0 y tiene una cola derecha muy larga "
       "(asimetría y curtosis altas). Esto confirma lo que muestran asimetría y curtosis y motiva evaluar `log1p` en la sección siguiente.")

### ¿Ayuda la transformación logarítmica?

Se compara la asimetría de cada variable antes y después de aplicar `log1p` (logaritmo de 1 + x, definido en cero). La decisión
formal se toma en el capítulo 8 con validación cruzada.

In [ ]:
cand = resumen_num[resumen_num["asimetría"].abs() > 1].index.tolist()
comparacion = pd.DataFrame({
    "asimetría original": [frame[c].skew() for c in cand],
    "asimetría con log1p": [np.log1p(frame[c].clip(lower=0)).skew() for c in cand]}, index=cand)
display(comparacion.round(2))
mejora = (comparacion["asimetría con log1p"].abs() < comparacion["asimetría original"].abs()).sum()
md_out(f"El logaritmo reduce la asimetría en **{mejora} de {len(cand)}** variables con |asimetría| > 1.")

## Variables categóricas

Se resume, para cada una, cuántos niveles tiene, qué porcentaje falta, cuál es la categoría más frecuente y cuántas categorías son
raras (menos de 1 % de las filas). Las de alta cardinalidad necesitan un tratamiento especial.

In [ ]:
rows = []
for c in cat_cols:
    s = frame[c]
    vc = s.value_counts(dropna=True)
    total = s.notna().sum()
    rows.append({"variable": c, "niveles": s.nunique(dropna=True), "faltantes_%": 100 * s.isna().mean(),
                 "categoría principal": str(vc.index[0]) if len(vc) else "—",
                 "principal_%": 100 * vc.iloc[0] / total if len(vc) else np.nan,
                 "categorías < 1 %": int((vc / total < 0.01).sum())})
resumen_cat = pd.DataFrame(rows).sort_values("niveles", ascending=False).reset_index(drop=True)
resumen_cat.to_csv(TABLES / "resumen_categoricas.csv", index=False)
display(resumen_cat.round(1))
alta = resumen_cat[resumen_cat["niveles"] > 30]["variable"].tolist()
md_out(f"Variables de **alta cardinalidad** (más de 30 niveles): {', '.join(f'`{v}`' for v in alta) if alta else 'ninguna'}. "
       "Los códigos de diagnóstico (`diag_1`, `diag_2`, `diag_3`) tienen cientos de niveles: se agrupan en categorías clínicas "
       "(circulatorio, respiratorio, digestivo, diabetes, etc.) siguiendo la clasificación CIE-9. Los identificadores codificados y la "
       "especialidad médica se tratan con una categoría «infrecuente» para los niveles raros.")

In [ ]:
elegidas = [c for c in ["race", "gender", "age", "admission_type_id", "discharge_disposition_id", "admission_source_id",
                        "max_glu_serum", "A1Cresult", "insulin", "change", "diabetesMed", "diag_1_grupo"] if c in frame.columns]
fig, axes = plt.subplots(4, 3, figsize=(14, 13))
for ax, c in zip(axes.ravel(), elegidas):
    (frame[c].astype("object").fillna("Missing").value_counts().head(8).iloc[::-1] / len(frame) * 100).plot.barh(ax=ax, color="#4c78a8")
    ax.set_title(c); ax.set_xlabel("% de encuentros")
for ax in axes.ravel()[len(elegidas):]:
    ax.axis("off")
plt.tight_layout(); save_figure(fig, "04_categoricas_principales.png"); plt.show()

In [ ]:
top_esp = frame["medical_specialty"].astype("object").fillna("Missing").value_counts()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
(top_esp.head(12).iloc[::-1] / len(frame) * 100).plot.barh(ax=axes[0], color="#f58518")
axes[0].set_title("medical_specialty: 12 categorías más frecuentes"); axes[0].set_xlabel("% de encuentros")
(frame["diag_1_grupo"].value_counts().iloc[::-1] / len(frame) * 100).plot.barh(ax=axes[1], color="#54a24b")
axes[1].set_title("Diagnóstico principal agrupado (CIE-9)"); axes[1].set_xlabel("% de encuentros")
plt.tight_layout(); save_figure(fig, "04_especialidad_y_diagnostico.png"); plt.show()
md_out(f"`medical_specialty` tiene {es(frame['medical_specialty'].nunique(), 0)} niveles y {es(100 * frame['medical_specialty'].isna().mean(), 1)} % de faltantes; "
       f"`diag_1` tiene {es(frame['diag_1'].nunique(), 0)} códigos distintos que se reducen a {frame['diag_1_grupo'].nunique()} grupos.")

### Medicamentos

Son 23 columnas con cuatro valores (`No`, `Steady`, `Up`, `Down`): si el medicamento se administró y si su dosis cambió durante
la estancia. La mayoría casi nunca se usa.

In [ ]:
meds = [c for c in frame.columns if set(frame[c].dropna().unique()) <= {"No", "Steady", "Up", "Down"} and frame[c].nunique() > 0]
uso = pd.DataFrame({"% con el medicamento (Steady/Up/Down)": [100 * (frame[c] != "No").mean() for c in meds],
                    "niveles": [frame[c].nunique() for c in meds]}, index=meds).sort_values("% con el medicamento (Steady/Up/Down)", ascending=False)
constantes = uso[uso["niveles"] <= 1].index.tolist()
casi = uso[(uso["% con el medicamento (Steady/Up/Down)"] < 1) & (uso["niveles"] > 1)].index.tolist()
display(uso.round(2))
md_out(f"- **Constantes** (un solo valor, sin información): {constantes if constantes else 'ninguna'}\n"
       f"- **Casi constantes** (menos de 1 % de uso): {len(casi)} medicamentos\n"
       f"- Los más usados son {', '.join(f'**{v}** ({es(uso.iloc[i, 0], 1)} %)' for i, v in enumerate(uso.index[:3]))}; el resto tiene poca variación.\n\n"
       "Las constantes se eliminan; las casi constantes se conservan pero sus niveles raros se agrupan como «infrecuente».")